In [1]:
from openai import OpenAI
client = OpenAI()

In [2]:
# --- Workflow Step Class ---
class WorkflowStep:
    def __init__(self, name, func):
        self.name = name
        self.func = func

    def run(self, data):
        print(f"Running step: {self.name}")
        return self.func(data)

class Workflow:
    def __init__(self, steps):
        self.steps = steps

    def execute(self, initial_data):
        data = initial_data
        for step in self.steps:
            data = step.run(data)
        return data

In [3]:
# --- Step 1: Summarization with LLM ---
def summarize_text(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # lightweight GPT-4 variant
        messages=[{"role": "user", "content": f"Summarize this in 3 bullet points:\n{text}"}],
        temperature=0.5
    )
    return response.choices[0].message.content

In [4]:
# --- Step 2: Sentiment Analysis with LLM ---
def analyze_sentiment(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Classify sentiment of this text as Positive, Neutral, or Negative:\n{text}"}],
        temperature=0
    )
    return response.choices[0].message.content

In [5]:
# --- Step 3: Post-process into JSON ---
def package_result(data):
    return {
        "summary": data["summary"],
        "sentiment": data["sentiment"]
    }

In [6]:
# --- Workflow Orchestration ---
def llm_workflow(user_input):
    # Step A: Summarize
    summary = summarize_text(user_input)
    # Step B: Sentiment
    sentiment = analyze_sentiment(user_input)
    # Step C: Package
    return package_result({"summary": summary, "sentiment": sentiment})

In [7]:
# --- Run Workflow ---
input_text = "AWS cloud architecture provides scalability, cost efficiency, and strong security compliance for enterprises."
output = llm_workflow(input_text)
print(output)

{'summary': '- **Scalability**: AWS cloud architecture allows enterprises to easily scale resources up or down based on demand.\n- **Cost Efficiency**: It offers a pay-as-you-go pricing model, helping businesses optimize their expenses.\n- **Strong Security Compliance**: AWS ensures robust security measures and compliance with industry standards to protect enterprise data.', 'sentiment': 'The sentiment of the text is Positive.'}


In [8]:
# Define workflow steps
def step1(data):
    return data.strip().lower()

def step2(data):
    return f"Processed: {data}"

def step3(data):
    return {"result": data, "length": len(data)}

In [9]:
# Create workflow
workflow = Workflow([
    WorkflowStep("Normalize Input", step1),
    WorkflowStep("Transform", step2),
    WorkflowStep("Package Result", step3)
])

In [10]:
# Run workflow
output = workflow.execute(" AWS cloud architecture provides scalability, cost efficiency, and strong security compliance for enterprises. ")
print(output)

Running step: Normalize Input
Running step: Transform
Running step: Package Result
{'result': 'Processed: aws cloud architecture provides scalability, cost efficiency, and strong security compliance for enterprises.', 'length': 120}
